# **Feature Engineering - Gold Price Forecasting**
Transform stationary return data into a forecasting-ready dataset by creating time-lagged features that enable prediction of future gold price movements without look-ahead bias.

---

## Core Strategy
1. Lag Feature Creation
    - Generate historical values (Lag 1–5 days) for each predictor (`SPX_Return`, `USO_Return`, `SLV_Return`, `EURUSD_Return`) and the target (`GLD_Return`).
    - Financial markets exhibit temporal dependency. Lagged features allow the model to learn patterns like "If oil dropped 3% yesterday, how does gold typically react today?"
    - 20 new lagged predictor columns + 5 autoregressive target lags.
2. Target Variable Definition
    - Shift the GLD_Return series by -1 to create a target column representing next-day return.
    - Ensures the model learns to predict future values using past/present information only. This eliminates look-ahead bias, a critical error in time-series forecasting.
    - A single target column aligned with lagged features.
3. Time-Based Train/Test Split
    - Split data chronologically (80% train, 20% test) without shuffling.
    - Preserves temporal order and simulates real-world deployment where future data is unavailable during training. Prevents data leakage and ensures valid performance estimation.
    - Two disjoint datasets with non-overlapping date ranges.
4. Artifact Persistence
    - Save `x_train`, `y_train`, `x_test`, `y_test`, and a `feature_manifest.json` to disk.
    - Enables reproducibility across model notebooks. The manifest documents exactly which features were engineered, when, and with what parameters—critical for auditing and iteration.
    - Serialized artifacts in `artifacts/FeatureSelection/` ready for model training.

# **Import Libraries**

In [1]:
# data manipulation
import pandas as pd
import numpy as np

# directory handling
import sys
import os

# file handling
from pathlib import Path
import joblib
import json

# logging
from datetime import datetime

# **Configurations**

In [2]:
# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# directoties
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "FeatureSelection"
DATA_DIR = PROJECT_ROOT / "data"

# Create directories if not exist
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# load gold price returns data
data_path = DATA_DIR / "gold_price_returns.csv"

df_returns = pd.read_csv(data_path, parse_dates=True, index_col=0)

In [5]:
# verify data loaded correctly
df_returns.head(10)

,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return
Date,,,,,
2008-01-03,0.000000,0.008367,-0.001274,0.006917,0.001902
2008-01-04,-0.024552,-0.005142,-0.013526,-0.007720,0.000679
2008-01-07,0.003223,-0.004229,-0.023412,-0.007516,-0.004875
2008-01-08,-0.018352,0.023711,0.007417,0.035674,0.060478
2008-01-09,0.013624,-0.002650,-0.010649,-0.004490,-0.058245
2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339
2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739
2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337
2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499


In [6]:
# lag features and target configuration
LAG_RANGE = range(1, 6)     # Lag 1-5 days
TARGET_SHIFT = -1           # Predict next day
TRAIN_RATIO = 0.8
TARGET_COL = "GLD_Return"
FEATURE_COLS = [col for col in df_returns.columns if col != 'GLD_Return']

#  **Create Lag Features**

In [7]:
df_engineered = df_returns.copy()

In [8]:
# create lags for features
for col in FEATURE_COLS:
    for lag in LAG_RANGE:
        df_engineered[f"{col}_lag{lag}"] = df_engineered[TARGET_COL].shift(lag)

In [9]:
# create aggressive lags
for lag in LAG_RANGE:
    df_engineered[f"{TARGET_COL}_lag{lag}"] = df_engineered[TARGET_COL].shift(lag)

In [10]:
# create target (next-day return)
df_engineered["target"] = df_engineered[TARGET_COL].shift(TARGET_SHIFT)

# drop NaN rows
df_engineered = df_engineered.dropna()

In [11]:
print("Shape:", df_engineered.shape)
print("Total Features:", df_engineered.shape[1]-1)      # exclude target

Shape: (2283, 31)
Total Features: 30


# **Time-Based Train/Test Split**

In [12]:
# split index for train and test dataset
split_idx = int(len(df_engineered) * TRAIN_RATIO)

train_df = df_engineered.iloc[:split_idx].copy()
test_df = df_engineered.iloc[split_idx:].copy()

In [13]:
# verify spliting
print(f"Train Dataset: {train_df.shape} - from  date: {train_df.index[0].date()} to {train_df.index[-1].date()}")
print(f"Test Dataset: {test_df.shape} - from date: {test_df.index[0].date()} to {test_df.index[-1].date()}")

Train Dataset: (1826, 31) - from  date: 2008-01-10 to 2016-04-13
Test Dataset: (457, 31) - from date: 2016-04-14 to 2018-05-14


# **Prepare Features and Target**

In [14]:
# get all lagged features
all_features = [col for col in train_df.columns if col not in [TARGET_COL, "target"] and "lag" in col]

In [15]:
# train-test split for final modeling
x_train, x_test, y_train, y_test = train_df[all_features], test_df[all_features], train_df["target"], test_df["target"]

In [16]:
# split verification
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

x_train shape: (1826, 25)
x_test shape: (457, 25)
y_train shape: (1826,)
y_test shape: (457,)


# **Save Artifacts**

In [17]:
# save train and test data in pkl format
artifacts = {
    "x_train" : x_train,
    "x_test" : x_test,
    "y_train" : y_train,
    "y_test" : y_test
}

for name, data in artifacts.items():
    joblib.dump(data, ARTIFACTS_DIR / f"{name}.pkl")

In [18]:
# save feature menifest
feature_manifest = {
    "created_at": datetime.now().isoformat(),
    "target_column": TARGET_COL,
    "base_features": FEATURE_COLS,
    "engineered_features": all_features,
    "lag_range": list(LAG_RANGE),
    "target_shift": TARGET_SHIFT,
    "train_samples": len(x_train),
    "test_samples": len(x_test),
    "date_range": {
        "train_start": train_df.index[0].isoformat(),
        "train_end": train_df.index[-1].isoformat(),
        "test_start": test_df.index[0].isoformat(),
        "test_end": test_df.index[-1].isoformat()
    }
}

with open (ARTIFACTS_DIR / "feature_menifest.json", "w") as f:
    json.dump(feature_manifest, f, indent=4)

In [19]:
# save preprocces train and test data
train_df.to_csv(DATA_DIR / "train_df.csv")
test_df.to_csv(DATA_DIR / "test_df.csv")

# **End of Feature Engineering**